In [ ]:
#整体志愿者heart_rate文件数据预处理
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

# 数据解析 
def parse_line(line):
    line = line.strip()
    # 替换错误的小数点分隔符为逗号（如"-556410.36066.57" → "-556410.36066,57"）
    if line.count(".") > 1:
        line = re.sub(r"\.(?=[^.]*\.)", ",", line)  # 只保留最后一个小数点
    return [float(x) for x in line.split(",")]

# 读取文件并处理数据(数据清洗)
data = []
with open("heart_rate", "r") as f:
    for line in f:
        if line.strip():  # 跳过空行
            try:
                seconds, bpm = parse_line(line)
                data.append({"seconds": seconds, "bpm": bpm})#将有效数据存储为字典列表，后续转换为DataFrame
            except:
                print(f"解析失败的行: {line}")#尝试解析每行数据，失败时打印警告（避免程序中断）

df = pd.DataFrame(data)
# 转换秒数为相对分钟数（从第一个时间点开始计算）
df["minutes"] = (df["seconds"] - df["seconds"].min()) / 60

# 过滤生理学不合理的心率值（正常范围30-200bpm）
df = df[(df["bpm"] >= 30) & (df["bpm"] <= 200)]
output_filename = "fixed_heart_rate.csv"
df.to_csv(output_filename, index=False) 

In [ ]:
#labels
import pandas as pd
import numpy as np
from IPython.display import display
df=pd.read_csv("labels.csv")
missing_values = df.isnull().sum()
print("各列缺失值统计：")
print(missing_values)
# 线性插值处理缺失值
df_interpolated = df.interpolate(method='linear')
data = pd.DataFrame(df)
# 再次检测缺失值
print("插值后各列缺失值统计：")
print(df_interpolated.isnull().sum())
display(df_interpolated.head())
output_filename_labels = "fixed_labels.csv"
df_interpolated.to_csv(output_filename_labels, index=False) 

各列缺失值统计：
time             0
sleep_state    438
user_id          0
dtype: int64
插值后各列缺失值统计：
time           0
sleep_state    0
user_id        0
dtype: int64


C:\\Users\\<user>\AppData\Local\Temp\ipykernel_17920\3280310985.py:9: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df_interpolated = df.interpolate(method='linear')


,time,sleep_state,user_id
0,0,0.0,1066528
1,30,0.0,1066528
2,60,0.0,1066528
3,90,0.0,1066528
4,120,0.0,1066528


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
df=pd.read_csv("motion.csv")
missing_values = df.isnull().sum()
print("各列缺失值统计：")
print(missing_values)
display(df.head())
output_filename_motion = "fixed_motion.csv"
df.to_csv(output_filename_motion, index=False)

各列缺失值统计：
time       0
x          0
y          0
z          0
user_id    0
dtype: int64


,time,x,y,z,user_id
0,-21684.846509,0.007080,0.000641,-0.987595,1066528
1,-21684.817053,0.004150,0.000626,-0.990555,1066528
2,-21684.807881,0.004150,0.001114,-0.990082,1066528
3,-21684.788503,0.005127,0.000626,-0.989090,1066528
4,-21684.768389,0.005615,0.000137,-0.988586,1066528


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
df=pd.read_csv("steps.csv")
missing_values = df.isnull().sum()
print("各列缺失值统计：")
print(missing_values)
display(df.head())
output_filename_steps = "fixed_steps.csv"
df.to_csv(output_filename_steps, index=False) 

各列缺失值统计：
time       0
steps      0
user_id    0
dtype: int64


,time,steps,user_id
0,-365207,243,1066528
1,-364607,578,1066528
2,-364007,848,1066528
3,-363407,166,1066528
4,-362807,0,1066528


In [ ]:
import pandas as pd

print("开始合并处理后的文件...")

# 1. 加载处理后的文件
try:
    df_hr = pd.read_csv("fixed_heart_rate.csv")
    df_labels = pd.read_csv("fixed_labels.csv")
    df_motion = pd.read_csv("fixed_motion.csv")
    df_steps = pd.read_csv("fixed_steps.csv")
    print("文件加载成功。")
except FileNotFoundError as e:
    print(f"错误：缺少文件 {e.filename}。请确保所有 fixed_*.csv 文件都已生成。")
    # 如果文件缺失，可以选择停止执行或进行其他处理
    raise e


print("排序数据...")
df_labels = df_labels.sort_values(by=['user_id', 'time'])
df_hr = df_hr.sort_values(by=['user_id', 'time'])
df_motion = df_motion.sort_values(by=['user_id', 'time'])
df_steps = df_steps.sort_values(by=['user_id', 'time'])
print("数据排序完成。")

print("开始合并数据 (merge_asof)...")
# 将心率合并到标签
merged_df = pd.merge_asof(
    df_labels,
    df_hr,
    on='time',
    by='user_id',
    direction='nearest',
    tolerance=15  # 容忍 15 秒的时间差
)
print("心率数据合并完成。")

merged_df = pd.merge_asof(
    merged_df,
    df_motion,
    on='time',
    by='user_id',
    direction='nearest',
    tolerance=15, # 容忍 15 秒的时间差
    suffixes=('', '_motion') # 如果有重名列（除了 on 和 by），添加后缀
)
print("运动数据合并完成。")

merged_df = pd.merge_asof(
    merged_df,
    df_steps,
    on='time',
    by='user_id',
    direction='nearest',
    tolerance=15, # 容忍 15 秒的时间差
    suffixes=('', '_steps') # 如果有重名列（除了 on 和 by），添加后缀
)
print("步数数据合并完成。")

# 检查合并后的缺失值情况
print("\n合并后各列缺失值统计：")
print(merged_df.isnull().sum())

#  保存最终合并的数据集
output_merged_filename = "cleaned_sleep_data.csv"
merged_df.to_csv(output_merged_filename, index=False)
print(f"\n最终合并的数据集已保存到: {output_merged_filename}")